# EduAssist AI: LLMOps – API Cost, Latency and Prompt Version Management

**Course:** MLOps – Experiential Learning Assignment 2

In this notebook I build a small question-answering pipeline for EduAssist AI and measure its latency, token usage,
cost and prompt versions.

| Task | What I do | Marks |
|---|---|---|
| Task 1 | Send 5 academic questions to the LLM and measure the latency of each call | 2 |
| Task 2 | Count input/output tokens and work out the cost per request and per month | 3 |
| Task 3 | Register 3 prompt versions in the MLflow Prompt Registry, compare them, and move the best one to `production` | 5 |

### Why I used a locally hosted model

A paid API such as OpenAI needs an account with billing enabled, which I did not have. Section 7 (Academic
Integrity) of the assignment allows a locally hosted open-source model instead, so I used
`Qwen/Qwen2.5-3B-Instruct`. It is an open, ungated model, and I ran it on my laptop with Hugging Face `transformers`.

I first tried the smaller `Qwen/Qwen2.5-1.5B-Instruct`. With prompt v3 it put every answer inside
```` ```json ```` code fences even though the prompt says not to, so its `valid_json` score was 0.0. A quick test
of the same 5 questions with the 3B model gave plain JSON without fences, so I used the 3B model for all three tasks.

How this maps to the assignment:

* The local `generate()` call is used in place of an API call, and I measure latency around it.
* Token counts come from the model's own tokenizer. They are the local equivalent of the `usage` counts an API returns.
* Running locally has no bill, so the cost is an estimate based on a published hosting price (Hugging Face
  Inference Endpoints).
* No account or API key is needed. If a token is ever needed, it is read from an environment variable.

## 0. Setup: imports and environment variables

I don't hard-code any secrets. `HF_TOKEN` is optional (it is only needed for gated models or faster downloads) and
is read from the environment or a local `.env` file. The cell only prints whether it is set, not its value.

In [1]:
import os
import json
import time
import difflib
import platform
import subprocess
from datetime import datetime, timezone

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
import transformers
import mlflow
from mlflow import MlflowClient
from dotenv import load_dotenv
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display

load_dotenv()
for var in ("HF_TOKEN", "MLFLOW_TRACKING_URI"):
    if os.environ.get(var) == "":
        os.environ.pop(var)

HF_TOKEN = os.getenv("HF_TOKEN")
print("HF_TOKEN set:           ", HF_TOKEN is not None)
print("MLFLOW_TRACKING_URI set:", os.getenv("MLFLOW_TRACKING_URI") is not None)

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)

HF_TOKEN set:            False
MLFLOW_TRACKING_URI set: False


## 1. Model configuration and loading

* `MODEL_NAME` is kept as one constant so the model is easy to change. The smaller `Qwen/Qwen2.5-1.5B-Instruct`
  or `Qwen/Qwen2.5-0.5B-Instruct` would run faster but are worse at following the JSON format. The first run
  downloads about 6 GB of weights into the Hugging Face cache.
* I use greedy decoding (`do_sample=False`, which is the same as temperature 0) and the same `MAX_NEW_TOKENS` for
  every prompt version. This way, differences between versions come from the prompt and not from random sampling.
* The device is chosen in this order: Apple Silicon `mps`, then `cuda`, then `cpu`. GPUs use `float16` and the CPU
  uses `float32`.
* Since there is no real API here, I record the versions of `transformers`, `torch` and `mlflow` as the
  "API version", along with the model revision (the Hugging Face commit hash).

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 512
DECODING = "greedy"
SEED = 42

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
DTYPE = torch.float16 if DEVICE in ("mps", "cuda") else torch.float32
torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, token=HF_TOKEN).to(DEVICE).eval()

# Remove the model's default sampling settings so generation is plain greedy decoding.
model.generation_config.do_sample = False
for attr in ("temperature", "top_p", "top_k"):
    setattr(model.generation_config, attr, None)
REPETITION_PENALTY = model.generation_config.repetition_penalty

MODEL_REVISION = getattr(model.config, "_commit_hash", None) or model_info(MODEL_NAME, token=HF_TOKEN).sha


def hardware_description():
    chip = platform.processor() or platform.machine()
    if platform.system() == "Darwin":
        try:
            chip = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                                  capture_output=True, text=True, check=True).stdout.strip()
        except Exception:
            pass
    elif DEVICE == "cuda":
        chip = torch.cuda.get_device_name(0)
    return f"{chip} ({platform.system()} {platform.machine()}), device={DEVICE}"


HARDWARE = hardware_description()
API_VERSION = f"transformers=={transformers.__version__}; torch=={torch.__version__}; mlflow=={mlflow.__version__}"

pd.Series({
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "parameters": f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f} B",
    "api_version (library versions)": API_VERSION,
    "python": platform.python_version(),
    "hardware": HARDWARE,
    "dtype": str(DTYPE).replace("torch.", ""),
    "decoding": DECODING,
    "max_new_tokens": MAX_NEW_TOKENS,
    "repetition_penalty (model default)": REPETITION_PENALTY,
}, name="value").to_frame()

W1105 22:38:52.740000 34068 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

,value
model_name,Qwen/Qwen2.5-3B-Instruct
model_revision,aa8e72537993ba99e69dfaafa59ed015b17504d1
parameters,3.09 B
api_version (library versions),transformers==5.22.1; torch==2.15.0; mlflow==3.17.2
python,3.12.6
hardware,"Apple M1 Pro (Darwin arm64), device=mps"
dtype,float16
decoding,greedy
max_new_tokens,512
repetition_penalty (model default),1.05


## 2. Shared inputs: five questions and three prompt templates

All tasks and all prompt versions use the same five questions. I picked questions from different subjects, some
needing a short definition and others a longer explanation, so that latency and token counts would vary.

The three prompt templates are copied exactly from the assignment. `{{question}}` (with double braces) is the
placeholder format used by the MLflow Prompt Registry.

In [3]:
QUESTIONS = [
    {"question_id": "Q1", "question": "What is overfitting in machine learning, and how can it be prevented?"},
    {"question_id": "Q2", "question": "State Newton's second law of motion."},
    {"question_id": "Q3", "question": "Explain how the quicksort algorithm works and state its average and worst-case time complexity."},
    {"question_id": "Q4", "question": "What is the difference between a p-value and a confidence interval in statistics?"},
    {"question_id": "Q5", "question": "Why does inflation reduce the purchasing power of money?"},
]

PROMPT_TEMPLATES = {
    1: """Answer the following student question clearly.
Question: {{question}}""",
    2: """You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}""",
    3: """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}""",
}


def render(template, question):
    return template.replace("{{question}}", question)


pd.DataFrame(QUESTIONS)

,question_id,question
0,Q1,"What is overfitting in machine learning, and how can it be prevented?"
1,Q2,State Newton's second law of motion.
2,Q3,Explain how the quicksort algorithm works and state its average and worst-case time complexity.
3,Q4,What is the difference between a p-value and a confidence interval in statistics?
4,Q5,Why does inflation reduce the purchasing power of money?


## 3. Helper function for calling the LLM

Every task uses the same function, `call_llm(prompt_text, question_id, prompt_version)`. It:

1. Wraps the prompt as a single user message using the model's chat template (`add_generation_prompt=True`).
   Qwen's chat template also adds its default system message, so those tokens are counted as input, the same way
   a system prompt would be billed by a hosted API.
2. Saves the UTC `start_time` and `end_time` and uses `time.perf_counter()` to time only `generate()` and decoding.
   Tokenisation is not timed, which is closer to the server-side time of an API call.
3. Counts tokens with the model's tokenizer: `input_tokens` is the prompt length after the chat template, and
   `output_tokens` is only the newly generated tokens.
4. Catches any exception and records `status="error"` with the error type, so one failed call does not stop the loop.

In [4]:
def call_llm(prompt_text, question_id, prompt_version):
    row = {
        "question_id": question_id,
        "prompt_version": f"v{prompt_version}",
        "model_name": MODEL_NAME,
        "start_time": None, "end_time": None, "latency_seconds": np.nan, "status": "error",
        "input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "hit_max_new_tokens": False,
        "response": "", "error": "",
    }
    start_time = datetime.now(timezone.utc)
    t0 = time.perf_counter()
    try:
        chat_text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(chat_text, return_tensors="pt").to(DEVICE)
        input_tokens = inputs["input_ids"].shape[-1]

        start_time = datetime.now(timezone.utc)
        t0 = time.perf_counter()
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id
            )
        response = tokenizer.decode(generated_ids[0, input_tokens:], skip_special_tokens=True)
        latency = time.perf_counter() - t0
        end_time = datetime.now(timezone.utc)

        output_tokens = generated_ids.shape[-1] - input_tokens
        row.update(
            latency_seconds=round(latency, 3), status="success", response=response,
            input_tokens=int(input_tokens), output_tokens=int(output_tokens),
            total_tokens=int(input_tokens + output_tokens),
            hit_max_new_tokens=bool(output_tokens >= MAX_NEW_TOKENS),
        )
    except Exception as exc:
        end_time = datetime.now(timezone.utc)
        row.update(latency_seconds=round(time.perf_counter() - t0, 3), error=type(exc).__name__)
    row["start_time"] = start_time.isoformat(timespec="milliseconds")
    row["end_time"] = end_time.isoformat(timespec="milliseconds")
    return row

## 4. Warm-up and throughput measurement (not part of the results)

The first generation after loading the model is slower because of one-time setup work (moving the weights,
compiling Metal/CUDA kernels and allocating memory). To keep this out of the Task 1 numbers, I run one warm-up call
first and don't record its time.

The same cell also measures two speeds that I need for pricing in the next section:

* Prefill throughput: input tokens processed per second in one forward pass. I use this for the input token price.
* Decode throughput: output tokens generated per second. I use this for the output token price.

In [5]:
def sync_device():
    if DEVICE == "mps":
        torch.mps.synchronize()
    elif DEVICE == "cuda":
        torch.cuda.synchronize()


warmup = call_llm(render(PROMPT_TEMPLATES[1], QUESTIONS[0]["question"]), "warmup", 1)
print(f"Warm-up call: status={warmup['status']}, latency={warmup['latency_seconds']} s (discarded)")

CALIBRATION_INPUT_TOKENS = 512
CALIBRATION_OUTPUT_TOKENS = 256
calib_text = " ".join(q["question"] for q in QUESTIONS) * 20
calib_ids = tokenizer(calib_text, return_tensors="pt")["input_ids"][:, :CALIBRATION_INPUT_TOKENS].to(DEVICE)

with torch.inference_mode():
    model(calib_ids)
    sync_device()
    t0 = time.perf_counter()
    for _ in range(5):
        model(calib_ids)
    sync_device()
    prefill_seconds = (time.perf_counter() - t0) / 5

    short_ids = calib_ids[:, :32]
    t0 = time.perf_counter()
    out = model.generate(short_ids, attention_mask=torch.ones_like(short_ids),
                         max_new_tokens=CALIBRATION_OUTPUT_TOKENS, min_new_tokens=CALIBRATION_OUTPUT_TOKENS,
                         do_sample=False, pad_token_id=tokenizer.pad_token_id)
    sync_device()
    decode_seconds = time.perf_counter() - t0

PREFILL_TOKENS_PER_SEC = CALIBRATION_INPUT_TOKENS / prefill_seconds
DECODE_TOKENS_PER_SEC = (out.shape[-1] - short_ids.shape[-1]) / decode_seconds
print(f"Prefill throughput: {PREFILL_TOKENS_PER_SEC:,.1f} input tokens/s")
print(f"Decode throughput:  {DECODE_TOKENS_PER_SEC:,.1f} output tokens/s")

Warm-up call: status=success, latency=24.878 s (discarded)


Prefill throughput: 589.9 input tokens/s
Decode throughput:  22.4 output tokens/s


## 5. Pricing configuration

I could not find a per-token price for `Qwen/Qwen2.5-3B-Instruct`. On the Hugging Face Hub, the only inference
provider for this model is Featherless AI, which charges a flat subscription instead of per token. So I worked out a
per-token price from a published hourly GPU price:

* Source: Hugging Face Inference Endpoints pricing, <https://huggingface.co/docs/inference-endpoints/en/pricing>
* Instance: AWS `nvidia-t4` x1 (one NVIDIA T4 with 14 GB). This is the smallest GPU option that can comfortably hold
  a 3B model in fp16 (about 6 GB of weights). Price: $0.50 per hour.
* Pricing date: 2026-11-02. Currency: USD.

A dedicated endpoint is billed by time, so the cost of a token is the fraction of an hour spent producing it:

$$\text{price per 1M tokens} = \frac{\text{hourly rate (USD)}}{\text{tokens per second} \times 3600} \times 1{,}000{,}000$$

I use the measured prefill speed for input tokens and the measured decode speed for output tokens. This assumes a
T4 endpoint would run at about the same speed as my laptop, with one request at a time. A real serving setup (for
example vLLM or TGI) handles many requests together, so the actual price per token would be lower and these numbers
are on the high side. The hourly rate is a plain constant in the code below and should be checked on the source page
before reusing it.

In [6]:
PRICING = {
    "pricing_reference": "Hugging Face Inference Endpoints - AWS nvidia-t4 x1 (1x NVIDIA T4)",
    "hourly_rate_usd": 0.50,
    "pricing_date": "2026-11-02",
    "currency": "USD",
    "pricing_source": "https://huggingface.co/docs/inference-endpoints/en/pricing",
}

INPUT_PRICE_PER_M = PRICING["hourly_rate_usd"] / (PREFILL_TOKENS_PER_SEC * 3600) * 1_000_000
OUTPUT_PRICE_PER_M = PRICING["hourly_rate_usd"] / (DECODE_TOKENS_PER_SEC * 3600) * 1_000_000
MONTHLY_REQUESTS = 10_000

pd.Series({
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "api_version (library versions)": API_VERSION,
    "device": DEVICE,
    **PRICING,
    "prefill_tokens_per_sec (measured)": round(PREFILL_TOKENS_PER_SEC, 1),
    "decode_tokens_per_sec (measured)": round(DECODE_TOKENS_PER_SEC, 1),
    "input_price_per_1M_tokens_usd (derived)": round(INPUT_PRICE_PER_M, 4),
    "output_price_per_1M_tokens_usd (derived)": round(OUTPUT_PRICE_PER_M, 4),
}, name="value").to_frame()

,value
model_name,Qwen/Qwen2.5-3B-Instruct
model_revision,aa8e72537993ba99e69dfaafa59ed015b17504d1
api_version (library versions),transformers==5.22.1; torch==2.15.0; mlflow==3.17.2
device,mps
pricing_reference,Hugging Face Inference Endpoints - AWS nvidia-t4 x1 (1x NVIDIA T4)
hourly_rate_usd,0.5
pricing_date,2026-11-02
currency,USD
pricing_source,https://huggingface.co/docs/inference-endpoints/en/pricing
prefill_tokens_per_sec (measured),589.9


---
# Task 1: Call the LLM and measure latency (2 marks)

For this baseline I use prompt v1 (basic) on all five questions.

In [7]:
task1_rows = [
    call_llm(render(PROMPT_TEMPLATES[1], q["question"]), q["question_id"], 1)
    for q in QUESTIONS
]
task1_df = pd.DataFrame(task1_rows)

LATENCY_COLS = ["question_id", "prompt_version", "model_name", "start_time", "end_time", "latency_seconds", "status"]
display(task1_df[LATENCY_COLS])

,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,status
0,Q1,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:09:53.091+00:00,2026-11-05T17:10:15.983+00:00,22.891,success
1,Q2,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:10:15.995+00:00,2026-11-05T17:10:24.047+00:00,8.051,success
2,Q3,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:10:24.087+00:00,2026-11-05T17:10:46.666+00:00,22.579,success
3,Q4,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:10:46.678+00:00,2026-11-05T17:11:09.293+00:00,22.615,success
4,Q5,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:11:09.308+00:00,2026-11-05T17:11:25.367+00:00,16.059,success


### Latency summary statistics

These are calculated over successful requests only. `std` is the sample standard deviation (ddof = 1).

In [8]:
ok = task1_df["status"] == "success"
latency_stats = task1_df.loc[ok, "latency_seconds"].agg(["mean", "median", "min", "max", "std"]).round(3)
display(latency_stats.to_frame("latency_seconds"))

slowest = task1_df.loc[task1_df.loc[ok, "latency_seconds"].idxmax()]
fastest = task1_df.loc[task1_df.loc[ok, "latency_seconds"].idxmin()]
for label, r in (("Slowest", slowest), ("Fastest", fastest)):
    print(f"{label}: {r.question_id} | latency={r.latency_seconds:.2f}s | input_tokens={r.input_tokens} | "
          f"output_tokens={r.output_tokens} | {r.output_tokens / r.latency_seconds:.1f} output tokens/s")
capped = task1_df.loc[task1_df["hit_max_new_tokens"], "question_id"].tolist()
print(f"Requests that hit the max_new_tokens={MAX_NEW_TOKENS} cap (answer truncated): {capped or 'none'}")
print(f"Pearson correlation (latency vs output_tokens): "
      f"{task1_df.loc[ok, 'latency_seconds'].corr(task1_df.loc[ok, 'output_tokens']):.3f}")

,latency_seconds
mean,18.439
median,22.579
min,8.051
max,22.891
std,6.480


Slowest: Q1 | latency=22.89s | input_tokens=54 | output_tokens=512 | 22.4 output tokens/s
Fastest: Q2 | latency=8.05s | input_tokens=46 | output_tokens=166 | 20.6 output tokens/s
Requests that hit the max_new_tokens=512 cap (answer truncated): ['Q1', 'Q3', 'Q4']
Pearson correlation (latency vs output_tokens): 0.999


### Interpretation

*The numbers below come from my recorded run on 2026-11-05 (Apple M1 Pro, MPS). Because decoding is greedy, the token counts stay the same on a re-run, but the latencies will change a little.*

The mean latency was 18.44 s (median 22.58 s, min 8.05 s, max 22.89 s, std 6.48 s). The slowest request was Q1
(overfitting) at 22.89 s, and the fastest was Q2 (Newton's second law) at 8.05 s.

Why some requests were slow:

* Output length is the main reason. The inputs were almost the same length (46 to 55 tokens), but Q1 produced 512
  output tokens while Q2 produced only 166. Latency and output tokens had a correlation of 0.999, and the speed
  stayed around 21 to 23 tokens per second. Tokens are generated one after another, so each one added about 45 ms.
* Q1, Q3 and Q4 all hit the 512-token limit. Prompt v1 doesn't say how long the answer should be, so for open
  questions like "explain" or "what is the difference" the model kept writing until it was cut off. These three are
  the requests between 22.58 and 22.89 s, and their answers are incomplete.
* Q1, Q3 and Q4 generated the same number of tokens, but Q1 still took about 0.3 s longer. Q1 was the first measured
  request, straight after the warm-up and throughput test, so I think the small difference is leftover setup work
  plus normal variation on my laptop GPU (background apps, memory use, temperature). With a hosted API, network
  delay and server load would cause similar variation.
* Cold start was left out. The warm-up call took 24.88 s; without it, the first measured request would have included
  this setup time and made the statistics look worse.

---
# Task 2: Count tokens and calculate cost (3 marks)

Token counting: a hosted API returns `usage.prompt_tokens`, `usage.completion_tokens` and `usage.total_tokens`.
Running locally, I get the same values from the model's tokenizer:

* `input_tokens`: number of `input_ids` after applying the chat template (same as `prompt_tokens`)
* `output_tokens`: generated length minus input length, so only the new tokens (same as `completion_tokens`)
* `total_tokens`: `input_tokens + output_tokens` (same as `total_tokens`)

Cost formulas (in USD, using the prices from Section 5):

* Input Cost = Input Tokens / 1,000,000 × Input Price per Million Tokens
* Output Cost = Output Tokens / 1,000,000 × Output Price per Million Tokens
* Total Cost per Request = Input Cost + Output Cost
* Projected Monthly Cost = Average Cost per Request × 10,000

In [9]:
def add_costs(df):
    df = df.copy()
    df["input_cost_usd"] = df["input_tokens"] / 1_000_000 * INPUT_PRICE_PER_M
    df["output_cost_usd"] = df["output_tokens"] / 1_000_000 * OUTPUT_PRICE_PER_M
    df["total_cost_usd"] = df["input_cost_usd"] + df["output_cost_usd"]
    return df


task2_df = add_costs(task1_df)
COST_COLS = ["question_id", "input_tokens", "output_tokens", "total_tokens",
             "input_cost_usd", "output_cost_usd", "total_cost_usd"]
display(task2_df[COST_COLS].style.format({c: "${:.6f}" for c in COST_COLS if c.endswith("_usd")}))

,question_id,input_tokens,output_tokens,total_tokens,input_cost_usd,output_cost_usd,total_cost_usd
0,Q1,54,512,566,$0.000013,$0.003180,$0.003193
1,Q2,46,166,212,$0.000011,$0.001031,$0.001042
2,Q3,55,512,567,$0.000013,$0.003180,$0.003193
3,Q4,53,512,565,$0.000012,$0.003180,$0.003193
4,Q5,48,367,415,$0.000011,$0.002280,$0.002291


In [10]:
avg_cost_per_request = task2_df.loc[ok, "total_cost_usd"].mean()
projected_monthly_cost = avg_cost_per_request * MONTHLY_REQUESTS
output_share = task2_df.loc[ok, "output_cost_usd"].sum() / task2_df.loc[ok, "total_cost_usd"].sum()

display(pd.Series({
    "avg_input_tokens": task2_df.loc[ok, "input_tokens"].mean(),
    "avg_output_tokens": task2_df.loc[ok, "output_tokens"].mean(),
    "avg_cost_per_request_usd": round(avg_cost_per_request, 6),
    "monthly_requests": MONTHLY_REQUESTS,
    "projected_monthly_cost_usd": round(projected_monthly_cost, 2),
    "output_tokens_share_of_tokens": f"{task2_df.loc[ok, 'output_tokens'].sum() / task2_df.loc[ok, 'total_tokens'].sum():.1%}",
    "output_tokens_share_of_cost": f"{output_share:.1%}",
}, name="value").to_frame())

,value
avg_input_tokens,51.2
avg_output_tokens,413.8
avg_cost_per_request_usd,0.002582
monthly_requests,10000
projected_monthly_cost_usd,25.82
output_tokens_share_of_tokens,89.0%
output_tokens_share_of_cost,99.5%


In [11]:
CSV_COLS = ["question_id", "prompt_version", "model_name", "start_time", "end_time", "latency_seconds", "status",
            "input_tokens", "output_tokens", "total_tokens", "hit_max_new_tokens",
            "input_cost_usd", "output_cost_usd", "total_cost_usd", "error"]
latency_tokens_cost = task2_df[CSV_COLS].copy()
latency_tokens_cost[["input_cost_usd", "output_cost_usd", "total_cost_usd"]] = \
    latency_tokens_cost[["input_cost_usd", "output_cost_usd", "total_cost_usd"]].round(8)
latency_tokens_cost.to_csv(os.path.join(OUTPUT_DIR, "latency_tokens_cost.csv"), index=False)
print("Saved", os.path.join(OUTPUT_DIR, "latency_tokens_cost.csv"))
latency_tokens_cost

Saved outputs/latency_tokens_cost.csv


,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,status,input_tokens,output_tokens,total_tokens,hit_max_new_tokens,input_cost_usd,output_cost_usd,total_cost_usd,error
0,Q1,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:09:53.091+00:00,2026-11-05T17:10:15.983+00:00,22.891,success,54,512,566,True,0.000013,0.003180,0.003193,
1,Q2,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:10:15.995+00:00,2026-11-05T17:10:24.047+00:00,8.051,success,46,166,212,False,0.000011,0.001031,0.001042,
2,Q3,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:10:24.087+00:00,2026-11-05T17:10:46.666+00:00,22.579,success,55,512,567,True,0.000013,0.003180,0.003193,
3,Q4,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:10:46.678+00:00,2026-11-05T17:11:09.293+00:00,22.615,success,53,512,565,True,0.000012,0.003180,0.003193,
4,Q5,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T17:11:09.308+00:00,2026-11-05T17:11:25.367+00:00,16.059,success,48,367,415,False,0.000011,0.002280,0.002291,


### Cost-optimisation recommendation

*The numbers below come from my recorded run on 2026-11-05 (Apple M1 Pro, MPS). Because decoding is greedy, the token counts stay the same on a re-run, but the latencies will change a little.*

What I found: an average v1 request used 51.2 input tokens and 413.8 output tokens and cost $0.002582. For 10,000
requests a month, that comes to about $25.82. Output tokens were 89.0% of all tokens but 99.5% of the cost. This is
because the output price I derived ($6.21 per 1M) is about 26 times the input price ($0.24 per 1M): generating tokens
one at a time keeps the GPU busy much longer than reading the input in one pass.

My recommendation is to make responses shorter, by asking for concise or structured answers and setting a lower
`max_new_tokens`. Input is only 0.5% of the cost, so reducing context or caching the fixed part of the prompt would
barely help here; output length is what matters. Three of the five v1 answers hit the 512-token limit, so we pay for
long answers that still get cut off. Task 3 shows this clearly: the JSON prompt v3 adds about 56 input tokens but
brings the average output down to 123.4 tokens. That lowers the cost to $0.000792 per request (69% less, about
$7.92 a month instead of $25.82), and latency drops by a similar amount. A lower limit such as 256 tokens would also
cap the worst-case cost and latency. Simple factual questions like Q2 could be sent to a smaller model (for example
Qwen2.5-1.5B), which runs faster and so would have a lower per-token price.

---
# Task 3: Prompt versioning and MLflow evaluation (5 marks)

* Tracking backend: `MLFLOW_TRACKING_URI` if it is set, otherwise a local SQLite file, `sqlite:///mlflow.db`. The
  Prompt Registry needs a database backend.
* Experiment name: `eduassist_llmops_assignment`.
* Prompt name: `eduassist-qa`. I register it three times to get versions 1, 2 and 3, each with a commit message,
  tags and the model settings used for evaluation.
* Avoiding duplicates: before registering a version, the notebook checks whether it already exists. If it exists
  with the same template, it is reused, so running the notebook again does not create extra versions. If it exists
  with a different template, the notebook stops and you need to start with a new `mlflow.db`.
* I set the run tags `mlflow.user` and `mlflow.source.name` to fixed values so my OS username and home folder path
  are not saved in MLflow.

In [12]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI") or "sqlite:///mlflow.db")
experiment = mlflow.set_experiment("eduassist_llmops_assignment")
client = MlflowClient()
print("Experiment:", experiment.name, "| id:", experiment.experiment_id)

PROMPT_NAME = "eduassist-qa"
COMMIT_MESSAGES = {
    1: "Initial basic prompt: plain answer to student question",
    2: "Add academic-assistant role, concise/accurate style, undergraduate-level language",
    3: "Enforce strict JSON output schema (answer, confidence, key_points) for downstream parsing",
}
PROMPT_TAGS = {
    1: {"style": "basic", "output_format": "free_text"},
    2: {"style": "structured", "output_format": "free_text"},
    3: {"style": "json_output", "output_format": "json"},
}
RUN_TAGS = {"mlflow.user": "eduassist-llmops", "mlflow.source.name": "eduassist_llmops.ipynb"}

for version, template in PROMPT_TEMPLATES.items():
    existing = mlflow.genai.load_prompt(PROMPT_NAME, version=version, allow_missing=True)
    if existing is not None:
        if existing.template != template:
            raise ValueError(f"{PROMPT_NAME} v{version} already exists with a different template. "
                             "Delete mlflow.db (fresh registry) or change PROMPT_NAME, then re-run.")
        print(f"v{version}: already registered with identical template - reusing")
        continue
    registered = mlflow.genai.register_prompt(
        name=PROMPT_NAME,
        template=template,
        commit_message=COMMIT_MESSAGES[version],
        tags={"project": "eduassist", "task": "academic_qa", **PROMPT_TAGS[version]},
        model_config={"model_name": MODEL_NAME, "max_tokens": MAX_NEW_TOKENS, "temperature": 0.0},
    )
    assert registered.version == version, f"Expected v{version}, registry created v{registered.version}"
    print(f"v{version}: registered - {COMMIT_MESSAGES[version]}")

registry_view = pd.DataFrame([
    {"version": v.version, "commit_message": v.commit_message,
     "tags": {k: t for k, t in v.tags.items() if not k.startswith("mlflow.")}}
    for v in (mlflow.genai.load_prompt(PROMPT_NAME, version=i) for i in PROMPT_TEMPLATES)
])
registry_view

2026/11/05 22:41:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/05 22:41:26 INFO mlflow.store.db.utils: Updating database tables


2026/11/05 22:41:27 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


Experiment: eduassist_llmops_assignment | id: 1
v1: registered - Initial basic prompt: plain answer to student question
v2: registered - Add academic-assistant role, concise/accurate style, undergraduate-level language
v3: registered - Enforce strict JSON output schema (answer, confidence, key_points) for downstream parsing


,version,commit_message,tags
0,1,Initial basic prompt: plain answer to student question,"{'project': 'eduassist', 'task': 'academic_qa', 'style': 'basic', 'output_format': 'free_text'}"
1,2,"Add academic-assistant role, concise/accurate style, undergraduate-level language","{'project': 'eduassist', 'task': 'academic_qa', 'style': 'structured', 'output_format': 'free_text'}"
2,3,"Enforce strict JSON output schema (answer, confidence, key_points) for downstream parsing","{'project': 'eduassist', 'task': 'academic_qa', 'style': 'json_output', 'output_format': 'json'}"


### Prompt diff (v1 → v2 → v3)

These diffs compare the templates after loading them back from the registry. The same comparison is available in
the MLflow UI under *Prompts → eduassist-qa → Compare*.

In [13]:
registered_templates = {v: mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{v}").template for v in (1, 2, 3)}
for a, b in ((1, 2), (2, 3)):
    print(f"===== diff v{a} -> v{b} =====")
    print("\n".join(difflib.unified_diff(
        registered_templates[a].splitlines(), registered_templates[b].splitlines(),
        fromfile=f"{PROMPT_NAME} v{a}", tofile=f"{PROMPT_NAME} v{b}", lineterm="")))
    print()

===== diff v1 -> v2 =====
--- eduassist-qa v1
+++ eduassist-qa v2
@@ -1,2 +1,4 @@
-Answer the following student question clearly.
+You are an academic assistant.
+Answer the following student question in a clear, concise, and accurate manner.
+Use simple language suitable for undergraduate students.
 Question: {{question}}

===== diff v2 -> v3 =====
--- eduassist-qa v2
+++ eduassist-qa v3
@@ -1,4 +1,10 @@
 You are an academic assistant.
-Answer the following student question in a clear, concise, and accurate manner.
-Use simple language suitable for undergraduate students.
+Answer the student question and return ONLY valid JSON.
+Do not include markdown, explanations outside JSON, or code fences.
+Use this exact JSON schema:
+{
+  "answer": "string",
+  "confidence": "high | medium | low",
+  "key_points": ["string"]
+}
 Question: {{question}}



### JSON validity check

$$\text{valid\_json} = \frac{\text{Valid JSON Responses}}{\text{Total Responses}}$$

A response is valid only if `json.loads` can parse the raw text and the result is an object with exactly these
keys: `answer` (a string), `confidence` (`high`, `medium` or `low`) and `key_points` (a list of strings). I don't
remove Markdown code fences before checking. The v3 prompt says not to use them and a downstream parser would fail on
them, so a fenced answer counts as invalid. I calculate the metric for all three versions, although v1 and v2 don't
ask for JSON, so I expect about 0 for them. Failed calls also count as invalid.

In [14]:
REQUIRED_KEYS = {"answer", "confidence", "key_points"}


def is_valid_json(text):
    try:
        obj = json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return False
    return (
        isinstance(obj, dict)
        and set(obj) == REQUIRED_KEYS
        and isinstance(obj["answer"], str)
        and obj["confidence"] in {"high", "medium", "low"}
        and isinstance(obj["key_points"], list)
        and all(isinstance(p, str) for p in obj["key_points"])
    )


assert is_valid_json('{"answer": "a", "confidence": "high", "key_points": ["x"]}')
assert not is_valid_json('```json\n{"answer": "a", "confidence": "high", "key_points": ["x"]}\n```')
assert not is_valid_json('{"answer": "a", "confidence": "very high", "key_points": ["x"]}')

### Evaluating each prompt version in a separate MLflow run

For each version, I load the prompt from the registry (`prompts:/eduassist-qa/<v>`, which also links the prompt
version to the run), fill it in with `prompt.format(question=...)` and run the same five questions. I then log the
parameters, the five required metrics, per-request step metrics and a table of responses as a run artifact.

In [15]:
METRIC_NAMES = ["avg_latency_seconds", "avg_input_tokens", "avg_output_tokens", "avg_cost_per_request", "valid_json"]
eval_frames, summary_rows = [], []

for version in (1, 2, 3):
    with mlflow.start_run(run_name=f"prompt_v{version}", tags=RUN_TAGS) as run:
        prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{version}")
        mlflow.log_params({
            "prompt_name": PROMPT_NAME,
            "prompt_version": prompt.version,
            "prompt_uri": f"prompts:/{PROMPT_NAME}/{prompt.version}",
            "model_name": MODEL_NAME,
            "model_revision": MODEL_REVISION,
            "api_version": API_VERSION,
            "device": DEVICE,
            "dtype": str(DTYPE).replace("torch.", ""),
            "decoding": DECODING,
            "max_new_tokens": MAX_NEW_TOKENS,
            "repetition_penalty": REPETITION_PENALTY,
            "num_questions": len(QUESTIONS),
            "input_price_per_1m_usd": round(INPUT_PRICE_PER_M, 6),
            "output_price_per_1m_usd": round(OUTPUT_PRICE_PER_M, 6),
            "pricing_hourly_rate_usd": PRICING["hourly_rate_usd"],
            "pricing_reference": PRICING["pricing_reference"],
            "pricing_date": PRICING["pricing_date"],
            "currency": PRICING["currency"],
            "pricing_source": PRICING["pricing_source"],
        })

        rows = []
        for step, q in enumerate(QUESTIONS):
            row = call_llm(prompt.format(question=q["question"]), q["question_id"], prompt.version)
            row["question"] = q["question"]
            row["valid_json"] = row["status"] == "success" and is_valid_json(row["response"])
            rows.append(row)
            mlflow.log_metrics({"request_latency_seconds": row["latency_seconds"],
                                "request_output_tokens": row["output_tokens"]}, step=step)

        df = add_costs(pd.DataFrame(rows))
        succ = df["status"] == "success"
        metrics = {
            "avg_latency_seconds": df.loc[succ, "latency_seconds"].mean(),
            "avg_input_tokens": df.loc[succ, "input_tokens"].mean(),
            "avg_output_tokens": df.loc[succ, "output_tokens"].mean(),
            "avg_cost_per_request": df.loc[succ, "total_cost_usd"].mean(),
            "valid_json": df["valid_json"].mean(),
        }
        mlflow.log_metrics(metrics)
        mlflow.log_metrics({"error_rate": 1 - succ.mean(), "truncation_rate": df["hit_max_new_tokens"].mean()})
        mlflow.log_table(
            df[["question_id", "question", "prompt_version", "status", "latency_seconds", "input_tokens",
                "output_tokens", "total_cost_usd", "valid_json", "response", "error"]],
            artifact_file=f"responses/prompt_v{version}_responses.json",
        )
        eval_frames.append(df)
        summary_rows.append({"prompt_version": f"v{version}", **metrics, "run_id": run.info.run_id})
        print(f"prompt_v{version}: " + ", ".join(f"{k}={v:.6g}" for k, v in metrics.items())
              + f", truncation_rate={df['hit_max_new_tokens'].mean():.2f}")

eval_df = pd.concat(eval_frames, ignore_index=True)

prompt_v1: avg_latency_seconds=18.8686, avg_input_tokens=51.2, avg_output_tokens=413.8, avg_cost_per_request=0.0025824, valid_json=0, truncation_rate=0.60


prompt_v2: avg_latency_seconds=12.0634, avg_input_tokens=73.2, avg_output_tokens=270.2, avg_cost_per_request=0.0016956, valid_json=0, truncation_rate=0.00


prompt_v3: avg_latency_seconds=5.6382, avg_input_tokens=107.2, avg_output_tokens=123.4, avg_cost_per_request=0.000791747, valid_json=1, truncation_rate=0.00


### Comparing responses (quality check)

The responses for each question and prompt version are shortened here. The full text is saved in each run's
`responses/` artifact.

In [16]:
preview = eval_df[["question_id", "prompt_version", "output_tokens", "hit_max_new_tokens", "valid_json", "response"]].copy()
preview["response"] = preview["response"].str.replace("\n", " ⏎ ").str.slice(0, 300)
preview.sort_values(["question_id", "prompt_version"]).reset_index(drop=True)

,question_id,prompt_version,output_tokens,hit_max_new_tokens,valid_json,response
0,Q1,v1,512,True,False,Overfitting in machine learning occurs when a model learns the detail and noise in the training data to an extent that it negatively impacts the performance...
1,Q1,v2,308,False,False,"Overfitting in machine learning happens when a model learns the training data too well, including its noise and random fluctuations. This means the model wo..."
2,Q1,v3,149,False,True,"{ ⏎ ""answer"": ""Overfitting occurs when a machine learning model performs well on training data but poorly on unseen test data due to the model being too c..."
3,Q2,v1,166,False,False,Newton's second law of motion states that the acceleration of an object as produced by a net force is directly proportional to the magnitude of the net forc...
4,Q2,v2,161,False,False,Newton's second law of motion is all about how force affects the movement of objects. It says that the force acting on an object is equal to the mass of tha...
5,Q2,v3,108,False,True,"{ ⏎ ""answer"": ""Newton's second law of motion states that the acceleration of an object as produced by a net force is directly proportional to the magnitud..."
6,Q3,v1,512,True,False,Certainly! Let's break down the explanation of the Quicksort algorithm and discuss its average and worst-case time complexities. ⏎ ⏎ ### How Quicksort Work...
7,Q3,v2,439,False,False,Sure! Let's talk about the quicksort algorithm and its time complexities. ⏎ ⏎ ### How Quicksort Works ⏎ ⏎ Quicksort is a popular sorting algorithm that us...
8,Q3,v3,113,False,True,"{ ⏎ ""answer"": ""The Quicksort algorithm is a divide-and-conquer algorithm that works by selecting a 'pivot' element from the array and partitioning the oth..."
9,Q4,v1,512,True,False,Certainly! Let's break down the concepts of p-values and confidence intervals to understand their differences and how they are used in statistical analysis....


In [17]:
for _, r in eval_df[eval_df["prompt_version"] == "v3"].iterrows():
    print(f"--- {r.question_id} | valid_json={r.valid_json} ---\n{r.response}\n")

--- Q1 | valid_json=True ---
{
  "answer": "Overfitting occurs when a machine learning model performs well on training data but poorly on unseen test data due to the model being too complex and capturing noise along with underlying patterns. To prevent overfitting, techniques such as regularization, increasing the amount of training data, using simpler models, and employing cross-validation can be applied.",
  "confidence": "high",
  "key_points": [
    "overfitting",
    "machine learning model",
    "training data",
    "test data",
    "model complexity",
    "noise",
    "underlying patterns",
    "regularization",
    "increasing training data",
    "simpler models",
    "cross-validation"
  ]
}

--- Q2 | valid_json=True ---
{
  "answer": "Newton's second law of motion states that the acceleration of an object as produced by a net force is directly proportional to the magnitude of the net force, in the same direction as the net force, and inversely proportional to the mass of the 

### Comparison table and how I pick the best version

EduAssist needs strict JSON output, so I rank the versions in this order:

1. Highest `valid_json`. If the output can't be parsed, the app can't use it, however good the answer is.
2. If tied, lowest `avg_cost_per_request`.
3. If still tied, lowest `avg_latency_seconds`.

In [18]:
comparison = pd.DataFrame(summary_rows)
ranked = comparison.sort_values(["valid_json", "avg_cost_per_request", "avg_latency_seconds"],
                                ascending=[False, True, True])
best_version = int(ranked.iloc[0]["prompt_version"].lstrip("v"))
comparison["recommended"] = np.where(comparison["prompt_version"] == f"v{best_version}", "Yes", "No")

RESULT_COLS = ["prompt_version", "avg_latency_seconds", "avg_input_tokens", "avg_output_tokens",
               "avg_cost_per_request", "valid_json", "recommended"]
results = comparison[RESULT_COLS].round({"avg_latency_seconds": 3, "avg_input_tokens": 1, "avg_output_tokens": 1,
                                         "avg_cost_per_request": 8, "valid_json": 3})
results.to_csv(os.path.join(OUTPUT_DIR, "prompt_evaluation_results.csv"), index=False)
print("Saved", os.path.join(OUTPUT_DIR, "prompt_evaluation_results.csv"))
print(f"Best version by rule: v{best_version}")

for _, r in comparison.iterrows():
    client.set_tag(r["run_id"], "recommended", r["recommended"])
results

Saved outputs/prompt_evaluation_results.csv
Best version by rule: v3


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request,valid_json,recommended
0,v1,18.869,51.2,413.8,0.002582,0.0,No
1,v2,12.063,73.2,270.2,0.001696,0.0,No
2,v3,5.638,107.2,123.4,0.000792,1.0,Yes


### Promoting the best version to the `production` alias

In [19]:
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=best_version)

production_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
assert production_prompt.version == best_version
print(f"prompts:/{PROMPT_NAME}@production -> version {production_prompt.version}")
print(f"commit message: {production_prompt.commit_message}\n")
print(production_prompt.template)

prompts:/eduassist-qa@production -> version 3
commit message: Enforce strict JSON output schema (answer, confidence, key_points) for downstream parsing

You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}


### Promotion justification

*The numbers below come from my recorded run on 2026-11-05 (Apple M1 Pro, MPS). Because decoding is greedy, the token counts stay the same on a re-run, but the latencies will change a little.*

I am promoting version 3 of `eduassist-qa` to the `production` alias.

| Version | Avg latency (s) | Avg input tokens | Avg output tokens | Avg cost (USD) | valid_json | Truncated |
|---|---|---|---|---|---|---|
| v1 | 18.87 | 51.2 | 413.8 | 0.002582 | 0.0 | 3 / 5 |
| v2 | 12.06 | 73.2 | 270.2 | 0.001696 | 0.0 | 0 / 5 |
| v3 | **5.64** | 107.2 | **123.4** | **0.000792** | **1.0** | 0 / 5 |

**JSON validity:** EduAssist's app needs strict JSON with `answer`, `confidence` and `key_points`. All five v3
responses parsed with `json.loads` and matched the schema (valid_json = 1.0, with no code fences). v1 and v2 gave
normal text with Markdown headings and lists (valid_json = 0.0), so the app could not use them at all. This alone
rules them out.

**Latency and tokens:** v3 was the fastest, at 5.64 s on average compared with 12.06 s for v2 and 18.87 s for v1
(about 3.3 times faster than v1). Its schema instructions add about 56 input tokens over v1, but because the output
has a fixed structure, the average output drops from 413.8 tokens (v1) and 270.2 (v2) to 123.4 tokens. None of the
v3 answers hit the 512-token limit, while 60% of the v1 answers were cut off.

**Cost:** Since output tokens make up most of the cost, v3 is also the cheapest at $0.000792 per request. That is
about $7.92 for 10,000 requests, compared with $25.82 for v1 and $16.96 for v2. The extra input tokens add only about
$0.000013 per request.

**Quality:** v3's answers were accurate and to the point for all five questions, although not always complete. v2 sometimes used informal examples
(ice cream to explain inflation and a "guessing game" for p-values). v3 is not perfect, though. For Q3 it left out the
average and worst-case time complexity that the question asked for, its `key_points` are short keywords rather than
full points, and it said `confidence: "high"` for every question, so the confidence value isn't very meaningful.

**Next steps:** In production I would keep the JSON check with a retry or fallback, and track valid_json and latency
for each release. I would also try a v4 prompt that asks the model to answer every part of multi-part questions and to
write key points as short sentences. It would be registered as a new version and only promoted if it does better than
v3 on the same questions.

---
## Outputs and where to find them in MLflow

* `outputs/latency_tokens_cost.csv`: per-request table from Task 1 and Task 2.
* `outputs/prompt_evaluation_results.csv`: comparison table from Task 3.
* MLflow UI: run `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000` and open <http://127.0.0.1:5000>.
  * *Experiments → eduassist_llmops_assignment*: runs `prompt_v1` to `prompt_v3` with their params, metrics and
    `responses/` artifacts.
  * *Prompts → eduassist-qa*: versions 1 to 3, commit messages, the compare view and the `production` alias.

In [20]:
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"{OUTPUT_DIR}/{f}  ({os.path.getsize(os.path.join(OUTPUT_DIR, f))} bytes)")

outputs/latency_tokens_cost.csv  (966 bytes)
outputs/prompt_evaluation_results.csv  (230 bytes)
